# Actividad 2 · Limitaciones de los modelos

**Tiempo estimado:** 20 minutos  
**Condición:** obligatoria

En esta actividad vas a analizar por qué un modelo puede producir resultados técnicamente válidos y, aun así, no ser suficiente para orientar una decisión. Trabajarás con errores de clasificación, diferencias entre grupos y un caso de uso relacionado con churn.

El propósito es reconocer los problemas y formular preguntas críticas antes de utilizar las predicciones.



## Preparación del entorno y reconstrucción del modelo

SQL selecciona los datos y Python reconstruye el mismo modelo introductorio utilizado en la actividad anterior.


In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

for package in ["duckdb", "pandas", "scikit-learn"]:
    module = "sklearn" if package == "scikit-learn" else package
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
DATA_PATH = Path("titanic.csv")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

titanic_original = pd.read_csv(DATA_PATH)
con = duckdb.connect(":memory:")
con.register("titanic_original", titanic_original)
con.execute("CREATE OR REPLACE TABLE titanic AS SELECT * FROM titanic_original")

print(f"Entorno preparado: {len(titanic_original):,} pasajeros y {titanic_original.shape[1]} variables.")


In [ ]:
df = con.sql("""
SELECT
    survived AS objetivo,
    pclass,
    sex,
    age,
    sibsp,
    fare
FROM titanic
""").df()

y = df["objetivo"]
X = df.drop(columns="objetivo")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Los valores faltantes se completan usando solo los datos de entrenamiento.
mediana_edad = X_train["age"].median()
X_train["age"] = X_train["age"].fillna(mediana_edad)
X_test["age"] = X_test["age"].fillna(mediana_edad)

# Las variables categóricas se preparan con las categorías de entrenamiento.
X_train = pd.get_dummies(X_train, columns=["sex"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["sex"], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
probabilidad = modelo.predict_proba(X_test)[:, 1]

print(f"Casos de entrenamiento: {len(X_train):,}")
print(f"Casos de prueba: {len(X_test):,}")


## Los errores no tienen siempre la misma consecuencia

En una clasificación binaria pueden aparecer dos situaciones diferentes:

- el modelo predice la clase positiva, pero el resultado real es negativo;
- el modelo predice la clase negativa, pero el resultado real es positivo.

En un problema de churn, ofrecer una acción de retención a una persona que no pensaba abandonar puede generar un costo innecesario. No detectar a una persona que efectivamente abandona puede implicar perderla. La importancia relativa de ambos errores depende del problema y no puede decidirse solamente con código.


In [ ]:
errores = pd.DataFrame({
    "grupo": X_test["sex_male"].map({False: "female", True: "male"}).to_numpy(),
    "resultado_real": y_test.to_numpy(),
    "prediccion": y_pred,
})
con.register("errores_modelo", errores)

resumen_errores = con.sql("""
SELECT
    resultado_real,
    prediccion,
    COUNT(*) AS cantidad
FROM errores_modelo
GROUP BY resultado_real, prediccion
ORDER BY resultado_real, prediccion
""").df()
display(resumen_errores)

resumen_grupos = con.sql("""
SELECT
    grupo,
    COUNT(*) AS casos,
    SUM(CASE WHEN resultado_real <> prediccion THEN 1 ELSE 0 END) AS errores
FROM errores_modelo
GROUP BY grupo
ORDER BY grupo
""").df()
display(resumen_grupos)


**Lectura cuidadosa:** esta comparación descriptiva permite detectar preguntas para investigar. No constituye por sí sola una evaluación de equidad ni demuestra que pertenecer a un grupo cause el resultado.


## Un modelo que predice no necesariamente resuelve el problema

Supongamos que una empresa identifica 300 clientes con riesgo de churn y les ofrece un beneficio de $20. La campaña cuesta $6.000. Solo 180 de esas personas realmente habrían abandonado el servicio y el beneficio logra retener a 90, que aportan $70 cada una. El ingreso preservado es $6.300 y el resultado neto estimado es apenas $300.

El modelo puede producir predicciones razonables y, sin embargo, la acción asociada puede tener un impacto escaso. Antes de decidir también deben analizarse costos, beneficios, capacidad operativa, riesgos y alternativas.


## Actividad de reflexión

Respondé dentro del notebook:

1. En un caso de churn, ¿qué error considerarías más importante y por qué?
2. ¿Qué variable relevante para el abandono podría no estar registrada en el dataset?
3. ¿Qué cambio de contexto podría hacer que un modelo entrenado con datos históricos pierda utilidad?
4. ¿Por qué una variable importante para predecir no necesariamente es una causa sobre la que convenga actuar?
5. Proponé una acción posible basada en una predicción de churn e indicá al menos dos recaudos previos.

## Cierre

Los modelos aprenden patrones de los datos disponibles. También heredan sus omisiones, errores y sesgos. Por eso, una decisión responsable debe combinar el resultado predictivo con conocimiento del problema, revisión crítica y seguimiento de sus consecuencias.


In [ ]:
con.close()
